# 5.1 · Packaging Agents with the Model Context Protocol (MCP)

An agent is only as useful as the tools it can reach. The Model Context Protocol (MCP) is the standard client-server interface between an agent and its tools, APIs and data sources: write a tool once as an MCP server, and any MCP-capable agent can use it.

In this notebook you will:

1. Build an MCP server with FastMCP.
2. Test its tools by hand with MCP Inspector.
3. Connect a LangGraph agent running Granite to it with `langchain-mcp-adapters`.

## Prerequisites

- Python 3.12+ and basic `async`/`await`
- Familiarity with LangGraph agents
- A Granite model: `REPLICATE_API_TOKEN` or Ollama running locally with `granite4.2:3b` pulled.

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain_ollama \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    mcp \
    langchain-mcp-adapters
! echo "::endgroup::"

# Steps

## Step 1. What MCP solves

Without a standard, each framework defines tools its own way, so a tool written for one framework has to be rewritten for another. MCP fixes the interface: any MCP client can call any MCP server.

An MCP server can expose 3 kinds of things:

| Primitive | What it is | Example |
|---|---|---|
| **Tools** | Functions the model can call | `find_flight` |
| **Resources** | Read-only data the client can load into context | A file, a database row |
| **Prompts** | Parameterized prompt templates | A summarization template |

There are 3 roles:

- **Host**: the application running the agent (here, this notebook).
- **Client**: a component inside the host that holds one connection to one server and speaks JSON-RPC 2.0.
- **Server**: a process that exposes tools, resources and prompts, either over `stdio` (local subprocess) or `streamable-http` (network).

```
Host (agent)
 └─ MCP client  <── JSON-RPC 2.0 over stdio or HTTP ──>  MCP server (tools, resources, prompts)
```

Every connection starts with a handshake: 

* the client sends `initialize`
* the server replies with its capabilities, and the client confirms with `initialized`.
* Only then can the client call `tools/list` or `tools/call`.

## Step 2. Build an MCP server with FastMCP

FastMCP turns a type-annotated Python function into an MCP tool with one decorator. 

It ships with the `mcp` SDK as `mcp.server.fastmcp`.

The server below exposes three travel tools backed by mock data: `find_flight`, `book_hotel` and `get_forecast`.

The cell writes it to `travel_mcp_server.py` so the client can start it as a separate process, the way a real MCP server runs.

In [ ]:
%%writefile travel_mcp_server.py
"""A minimal MCP server exposing travel-planning tools, built with FastMCP."""

import re
from datetime import date, timedelta

from mcp.server.fastmcp import FastMCP

mcp = FastMCP(name="travel-assistant")

# Users and models write the same city many ways: "Atlanta", "ATLANTA", "ATL", "Atlanta, GA".
# Map every spelling to one canonical name before looking up the mock data.
CITY_ALIASES = {"atlanta": "Atlanta", "atl": "Atlanta"}

# Mock backend data (stand-ins for real flight/hotel/weather APIs)
MOCK_FLIGHTS = {
    "Atlanta": {"flight": "UA321", "departs": "08:30", "arrives": "10:00", "price_usd": 285},  # daily
}
MOCK_HOTELS = {
    "Atlanta": {"hotel": "Atlanta Grand Hotel", "price_per_night_usd": 195},
}
MOCK_WEATHER = {
    "Atlanta": {
        "2026-10-25": "Partly cloudy, 22C with a light breeze",
        "2026-10-26": "Sunny, 24C",
        "2026-10-27": "Showers in the afternoon, 19C",
    },
}


def normalize_city(destination: str) -> str | None:
    """Return the canonical city name for a destination, or None if unknown."""
    for word in re.findall(r"[a-z]+", destination.lower()):
        if word in CITY_ALIASES:
            return CITY_ALIASES[word]
    return None


@mcp.tool()
def find_flight(destination: str, date: str) -> dict:
    """Search for available flights to a destination (city name or airport code) on a date (YYYY-MM-DD)."""
    city = normalize_city(destination)
    if city not in MOCK_FLIGHTS:
        return {"error": f"No flights found to {destination}"}
    return {"destination": city, "date": date, **MOCK_FLIGHTS[city]}


@mcp.tool()
def book_hotel(destination: str, check_in: str, nights: int) -> dict:
    """Book a hotel at a destination (city name or airport code) for a number of nights from check_in (YYYY-MM-DD).

    A trip of N days needs N-1 nights (a 2-day trip is 1 night).
    """
    city = normalize_city(destination)
    if city not in MOCK_HOTELS:
        return {"error": f"No hotels available in {destination}"}
    hotel = MOCK_HOTELS[city]
    return {
        "destination": city,
        "check_in": check_in,
        "nights": nights,
        **hotel,
        "total_usd": hotel["price_per_night_usd"] * nights,
        "confirmation": f"HT-{city.upper()}-{check_in}",
    }


@mcp.tool()
def get_forecast(destination: str, start_date: str, days: int = 1) -> dict:
    """Get the daily weather forecast for a destination (city name or airport code).

    Returns one forecast per day for `days` days from start_date (YYYY-MM-DD). Use the trip length for `days`.
    """
    city = normalize_city(destination)
    forecasts = MOCK_WEATHER.get(city)
    if not forecasts:
        return {"error": f"No forecast available for {destination}"}
    start = date.fromisoformat(start_date)
    dates = [(start + timedelta(days=i)).isoformat() for i in range(days)]
    return {
        "destination": city,
        "daily": [{"date": d, "forecast": forecasts.get(d, "No forecast available")} for d in dates],
    }


if __name__ == "__main__":
    # Runs the server over stdio transport by default -- ideal for local
    # subprocess-based clients such as the one LangGraph will use below.
    mcp.run()

FastMCP built each tool's JSON Schema from its signature, type hints and docstring. The model reads those docstrings to decide which tool to call, so write them for the model.

### Step 2.1. Test the server with MCP Inspector

[MCP Inspector](https://github.com/modelcontextprotocol/inspector) is a browser UI that connects to any MCP server so you can call its tools by hand, with no agent or model involved. Test a server this way before you give it to an agent: if a tool misbehaves here, the problem is in the server, not in the model.

MCP Inspector is already running: open [http://localhost:6274](http://localhost:6274).

The server must run with a Python interpreter that has `mcp` installed. The next cell prints the exact values for the notebook's own environment.

In the Inspector, select **Add server** and fill in the form with the values printed above:

| Field | Value |
|---|---|
| Server ID | `travel` |
| Transport | stdio (local process) |
| Command | The `Command` path printed above. The grey `npx` in the form is only a placeholder; leaving it there gives "Command is required". |
| Arguments | `travel_mcp_server.py` (one argument per line) |
| Environment | Leave empty |
| Working directory | The `Working directory` path printed above |

Use the full interpreter path, not `npx` or plain `python`: `npx` launches servers published as npm packages, and plain `python` may resolve to an interpreter without `mcp`.

Then:

1. Select **Connect**, open **Tools**, and select **List Tools**. You should see `find_flight`, `book_hotel` and `get_forecast`, with the schemas FastMCP generated.
2. Run `find_flight` with `destination=Atlanta` and `date=2026-10-25`. It returns flight `UA321`.
3. Run it again with `destination=ATL`. It returns the same flight, because the server normalizes the city.
4. Run it with `destination=Paris`. It returns the error message from the mock data.

## Step 3. Connect a LangGraph agent

### Connect to Granite with `get_llm()`

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
import requests
from ibm_granite_community.notebook_utils import get_env_var


def ollama_has_model(host: str, model: str) -> bool:
    """True only if `host` answers AND `model` is already pulled there.

    Checking reachability alone is not enough: a running Ollama without the
    requested tag would otherwise trigger an on-demand pull, which needs
    network access we don't have on the day.
    """
    try:
        tags = requests.get(f"{host}/api/tags", timeout=2).json()
        return model in {m["name"] for m in tags.get("models", [])}
    except requests.RequestException:
        return False


def get_llm(
    model: str | None = None,
    *,
    backend: str | None = None,
    stop: list[str] | None = None,
    repetition_penalty: float | None = None,
    **kwargs,
):
    """Return a LangChain chat model for Granite.

    Args:
        model: An Ollama tag (e.g. "granite4.2:3b") or a Replicate model path
            (e.g. "ibm-granite/granite-4.2-8b"). Defaults to the `GRANITE_MODEL`
            env var (or "granite4.2:3b") for Ollama, and to `REPLICATE_MODEL`
            (or "ibm-granite/granite-4.2-8b") for Replicate.
        backend: Force "ollama" or "replicate"; omit to auto-resolve
            (Replicate if `REPLICATE_API_TOKEN` is set in the environment,
            else Ollama if the tag is pulled locally, Replicate otherwise).
        stop: Stop sequences, passed the way each backend expects them.
        repetition_penalty: Penalty for repeated tokens, passed the way each
            backend expects it.
        **kwargs: Extra keyword arguments passed through to the chat model.
    """
    host = get_env_var("OLLAMA_HOST", "http://127.0.0.1:11434")
    ollama_model = model or get_env_var("GRANITE_MODEL", "granite4.2:3b") or "granite4.2:3b"

    if backend is None:
        # A Replicate token in the environment is an explicit choice, so it wins over a local Ollama.
        if os.environ.get("REPLICATE_API_TOKEN"):
            backend = "replicate"
        elif ollama_has_model(host, ollama_model):
            backend = "ollama"

    if backend == "ollama":
        from langchain_ollama import ChatOllama

        return ChatOllama(
            model=ollama_model,
            base_url=host,
            num_predict=8192,  # Set the maximum number of tokens to generate as output.
            stop=stop,
            repeat_penalty=repetition_penalty,
            **kwargs,
        )

    from langchain_replicate import ChatReplicate

    # Only a Replicate model path ("owner/name") can be sent to Replicate; an Ollama tag cannot.
    if model and "/" in model:
        replicate_model = model
    else:
        replicate_model = get_env_var("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b") or "ibm-granite/granite-4.2-8b"

    model_kwargs = {
        "max_completion_tokens": 8192,  # Set the maximum number of tokens to generate as output.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    if stop:
        model_kwargs["stop"] = stop
    if repetition_penalty is not None:
        model_kwargs["repetition_penalty"] = repetition_penalty

    return ChatReplicate(
        model=replicate_model,
        replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
        model_kwargs=model_kwargs,
        **kwargs,
    )

llm = get_llm()
print(f"Connected via {type(llm).__name__}, model={llm.model!r}")

`MultiServerMCPClient` from `langchain-mcp-adapters` starts the server as a `stdio` subprocess with the notebook's own Python interpreter, runs the MCP handshake, and converts the server's tools into LangChain tools that a LangGraph agent uses like any other tool.

In [ ]:
import sys

mcp_client = MultiServerMCPClient(
    {
        "travel": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["travel_mcp_server.py"],
        }
    }
)

# Discover tools exposed by the MCP server -- this performs the full MCP
# handshake (initialize -> initialized -> tools/list) behind the scenes.
mcp_tools = await mcp_client.get_tools()

print(f"Discovered {len(mcp_tools)} MCP tools:")

for t in mcp_tools:
    print(f" - {t.name}: {t.description}")

In [ ]:
# Build a LangGraph ReAct-style agent equipped with our MCP tools.
travel_agent = create_agent(llm, mcp_tools)

async def run_agent(query: str):
    result = await travel_agent.ainvoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

response = await run_agent(
    "Plan a 2-day trip to Atlanta starting 2026-10-25. Find me a flight, book a hotel, "
    "and tell me the weather so I can pack appropriately."
)
print(response)


What happened:

1. `MultiServerMCPClient` started `travel_mcp_server.py` as a subprocess and connected over `stdio`.
2. `get_tools()` ran the handshake and turned the `tools/list` response into LangChain tools.
3. `create_agent` bound those tools to the Granite model.
4. Each time the model asked for a tool, LangGraph sent a `tools/call` request to the server and passed the result back to the model.

The agent code contains no protocol code.

## Step 4. When to use MCP

Use MCP when:

- Several agents or frameworks need the same tool.
- A different team owns and versions the tool than the agent.
- The tool must run in its own process or on another machine.

For a tool that only one agent uses in one process, a plain LangChain tool is simpler.

MCP connects an agent to tools. 

To let agents delegate work to other agents, see [5.2 · Packaging Agents with Agent2Agent (A2A)](06_packaging_a2a.ipynb).

## Step 5. Package the whole agent as an MCP server

So far the MCP server exposed *tools*, and the agent lived in this notebook. You can go one level up: wrap the agent itself in an MCP server, so any MCP client (another agent, an IDE, Claude Desktop, MCP Inspector) can hand it a whole task.

The packaged server exposes a single tool, `plan_trip`. Behind it:

```
MCP client ──tools/call plan_trip──> travel_agent_mcp_server.py (Granite agent)
                                         └─ MCP client ──tools/call find_flight, book_hotel, get_forecast──> travel_mcp_server.py
```

The agent server is an MCP *server* to its caller and an MCP *client* of the tools server. The caller sees one tool and never learns that a model, a ReAct loop and three more tools sit behind it.

The cell below writes it to `travel_agent_mcp_server.py`. It is the agent from step 3, moved into a FastMCP tool:

- It builds its own Granite model from environment variables (or `.env`). It can't use `get_llm()` from this notebook because it runs as a separate process.
- It writes nothing to stdout: over `stdio`, stdout carries the JSON-RPC messages, so a stray `print` breaks the connection.
- It builds the agent on the first call, not at startup, so `initialize` and `tools/list` answer instantly.


In [ ]:
%%writefile travel_agent_mcp_server.py
"""The travel agent from step 3, packaged as an MCP server with a single `plan_trip` tool."""

import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.server.fastmcp import FastMCP

# Load REPLICATE_API_TOKEN / GRANITE_MODEL / OLLAMA_HOST from a .env next to (or above) this file.
load_dotenv(find_dotenv())

HERE = Path(__file__).resolve().parent

mcp = FastMCP(name="travel-agent")

# This server is itself an MCP client of the tools server from step 2.
tools_client = MultiServerMCPClient(
    {
        "travel": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(HERE / "travel_mcp_server.py")],
        }
    }
)


def get_llm():
    """Granite via Replicate if REPLICATE_API_TOKEN is set, else via a local Ollama."""
    if os.environ.get("REPLICATE_API_TOKEN"):
        from langchain_replicate import ChatReplicate

        return ChatReplicate(
            model=os.environ.get("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b"),
            model_kwargs={"max_completion_tokens": 8192, "chat_template_kwargs": {"enable_thinking": False}},
        )

    from langchain_ollama import ChatOllama

    return ChatOllama(
        model=os.environ.get("GRANITE_MODEL", "granite4.2:3b"),
        base_url=os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434"),
        num_predict=8192,
    )


_agent = None


async def get_agent():
    """Build the agent once, on the first call, so the server starts instantly."""
    global _agent
    if _agent is None:
        _agent = create_agent(get_llm(), await tools_client.get_tools())
    return _agent


@mcp.tool()
async def plan_trip(request: str) -> str:
    """Plan a trip end to end: find a flight, book a hotel and check the weather, then summarize.

    `request` is a plain-language trip request with the destination, start date (YYYY-MM-DD) and trip length.
    """
    agent = await get_agent()
    result = await agent.ainvoke({"messages": [{"role": "user", "content": request}]})
    # `.text` flattens the reply to a string: some backends return a list of content blocks.
    return result["messages"][-1].text


if __name__ == "__main__":
    mcp.run()


### Step 5.1. Call the packaged agent from any MCP client

Connect to the new server exactly as you connected to the tools server in step 3. This time `tools/list` returns one tool, and calling it runs the whole agent. No model, prompt or agent code is needed on the calling side.


In [ ]:
# A stdio server only inherits a few variables (PATH, HOME, ...) by default, so pass the
# environment to give it the same Granite backend as this notebook. In Colab the Replicate
# token lives in secrets, not in the environment: copy it there first.
if type(llm).__name__ == "ChatReplicate":
    os.environ["REPLICATE_API_TOKEN"] = get_env_var("REPLICATE_API_TOKEN")

agent_client = MultiServerMCPClient(
    {
        "travel_agent": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["travel_agent_mcp_server.py"],
            "env": dict(os.environ),
        }
    }
)

agent_tools = await agent_client.get_tools()
for t in agent_tools:
    print(f" - {t.name}: {t.description}")

plan_trip = agent_tools[0]
result = await plan_trip.ainvoke(
    {
        "request": "Plan a 2-day trip to Atlanta starting 2026-10-25. Find me a flight, book a hotel, "
        "and tell me the weather so I can pack appropriately."
    }
)
# MCP tool results come back as a list of content blocks: print the text ones.
print("\n".join(block["text"] for block in result if block["type"] == "text"))


### Step 5.2. Call the packaged agent from MCP Inspector

The simplest way to see that the agent is now a plain MCP server is to call it from MCP Inspector, the same way you called `find_flight` in step 2.1:

1. Add a second server in the Inspector, with Server ID `travel-agent`, transport `stdio`, the same **Command** as in step 2.1, and **Arguments** `travel_agent_mcp_server.py` (or its full path).
2. Connect, open **Tools**, and select **List Tools**. You should see one tool, `plan_trip`, not the three travel tools.
3. Run `plan_trip` with `request` set to:

    ```text
    Plan a 2-day trip to Atlanta starting 2026-10-25. Find me a flight, book a hotel, and tell me the weather.
    ```

    It returns the agent's trip summary. The Inspector shows a single `tools/call`: the model calls and the three tool calls all happen inside the server.

> **Timeout:** the Inspector cancels a request after 10 seconds by default, and an agent run takes longer. Before you run `plan_trip`, raise **Request Timeout** in the Inspector settings to `120000` (ms).

> **Backend:** the server reads `REPLICATE_API_TOKEN` from the environment or from the `.env` file in the repository. Without it, it uses Ollama with `granite4.2:3b`.


What changed compared with step 3:

| | Step 3: tools as MCP | Step 5: agent as MCP |
|---|---|---|
| What the server exposes | `find_flight`, `book_hotel`, `get_forecast` | `plan_trip` |
| Where the model runs | In the caller | Inside the server |
| What the caller needs | A model and an agent loop | Only an MCP client |

Package the agent as an MCP server when the caller should delegate a whole task and doesn't need to see the steps. MCP still treats it as a single tool call, though: it has no notion of long-running tasks, progress or multi-turn conversation between agents. For agent-to-agent collaboration, see [5.2 · Packaging Agents with Agent2Agent (A2A)](06_packaging_a2a.ipynb).


## Summary

- MCP separates a tool's implementation from the agents that use it.
- FastMCP generates tool schemas from Python functions. The docstrings are what the model reads.
- `langchain-mcp-adapters` lets a LangGraph agent use MCP tools without any protocol code.
- Every MCP session follows `initialize` → `tools/list` → `tools/call`. Knowing this sequence helps you debug.

### Resources

- [Model Context Protocol specification](https://modelcontextprotocol.io/)
- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
- [LangChain MCP adapters](https://reference.langchain.com/python/langchain-mcp-adapters)
- [LangGraph documentation](https://langchain-ai.github.io/langgraph/)